# THYROID_2026 — Statistical Analysis Examples

Interactive companion to `scripts/36_statistical_analysis_examples.py`.  
Demonstrates all `ThyroidStatisticalAnalyzer` capabilities against the publication-ready MotherDuck lakehouse.

| Section | Content |
|---------|--------|
| 1. Connection | MotherDuck / local DuckDB setup |
| 2. Table 1 | Cohort description with SMD |
| 3. Hypothesis Tests | FDR-corrected battery |
| 4. Logistic Regression | OR table + VIF + snippet |
| 5. Cox PH | HR table + forest plot |
| 6. Longitudinal | Tg/TSH mixed-effects trajectory |
| 7. Power Analysis | Sample-size calculations |
| 8. Correlation | Spearman matrix + heatmap |

**Random seed**: 42  
**Data source priority**: `risk_enriched_mv` → `advanced_features_v3` → `ptc_cohort`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

np.random.seed(42)
pio.renderers.default = 'notebook'

# Ensure project root on path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.statistical_analysis import (
    ThyroidStatisticalAnalyzer,
    THYROID_PREDICTORS,
    THYROID_SURVIVAL,
    THYROID_NSQIP_OUTCOMES,
    THYROID_NSQIP_PREDICTORS,
    LONGITUDINAL_MARKERS,
    ETE_SUBTYPES,
)

print('Imports OK')

## 1. Connect to MotherDuck (or local DuckDB)

In [ ]:
import os
from motherduck_client import MotherDuckClient, MotherDuckConfig

USE_LOCAL = os.getenv('USE_LOCAL_DUCKDB', '').lower() in ('1', 'true', 'yes')

if USE_LOCAL:
    import duckdb
    local_path = os.getenv('LOCAL_DUCKDB_PATH', str(ROOT / 'thyroid_master_local.duckdb'))
    con = duckdb.connect(local_path)
    print(f'Local DuckDB: {local_path}')
else:
    cfg = MotherDuckConfig(database='thyroid_research_2026')
    con = MotherDuckClient(cfg).connect_rw()
    print('Connected to MotherDuck: thyroid_research_2026')

analyzer = ThyroidStatisticalAnalyzer(con)
print('ThyroidStatisticalAnalyzer ready')

In [ ]:
# Resolve best available analytic view
resolved = analyzer.resolve_view()
print(f'Primary view: {resolved}')

df = con.execute(f'SELECT * FROM {resolved}').fetchdf()
print(f'Loaded {len(df):,} rows × {len(df.columns)} columns')
df.head(3)

## 2. Table 1 — Cohort Description with SMD

In [ ]:
# Overall Table 1
t1_overall, meta_overall = analyzer.generate_table_one(data=df)
print(f"N={meta_overall['n_total']:,} | "
      f"{len(meta_overall.get('continuous_vars',[]))} continuous, "
      f"{len(meta_overall.get('categorical_vars',[]))} categorical")
t1_overall

In [ ]:
# Table 1 grouped by sex — includes SMD for balance assessment
groupby_col = 'sex' if 'sex' in df.columns else None
t1_sex, meta_sex = analyzer.generate_table_one(data=df, groupby_col=groupby_col)
print(f"SMD computed: {meta_sex.get('smd_computed', False)}")
t1_sex

In [ ]:
# Table 1 by BRAF status
if 'braf_positive' in df.columns:
    t1_braf, _ = analyzer.generate_table_one(data=df, groupby_col='braf_positive')
    display(t1_braf)
else:
    print('braf_positive not in this view')

## 3. Hypothesis Testing Battery (FDR-corrected)

In [ ]:
target = 'event_occurred' if 'event_occurred' in df.columns else 'braf_positive'
features = [p for p in THYROID_PREDICTORS if p in df.columns and p != target]

ht_results = analyzer.run_hypothesis_tests(
    data=df,
    target_var=target,
    feature_list=features,
    correction='fdr_bh',
)

n_sig = int(ht_results['significant'].sum()) if 'significant' in ht_results.columns else 0
print(f"{len(ht_results)} tests | {n_sig} significant (FDR p<0.05)")
ht_results.style.highlight_between(subset=['p_adjusted'], left=0, right=0.05, color='#1a8a7a33')

## 4. Logistic Regression

In [ ]:
outcome = 'event_occurred' if 'event_occurred' in df.columns else 'braf_positive'
predictors = [
    p for p in THYROID_PREDICTORS
    if p in df.columns and p != outcome
    and pd.api.types.is_numeric_dtype(df[p])
][:8]

lr_result = analyzer.fit_logistic_regression(
    outcome=outcome,
    predictors=predictors,
    data=df,
)

if 'error' in lr_result:
    print('ERROR:', lr_result['error'])
else:
    print(f"N={lr_result['n_obs']:,} | pseudo-R²={lr_result['pseudo_r2']:.4f} | AUC={lr_result.get('auc','N/A')}")
    for w in lr_result.get('warnings', []):
        print('⚠', w)
    lr_result['or_table']

In [ ]:
# Forest plot (OR)
if 'error' not in lr_result:
    or_table = lr_result['or_table']
    forest_df = or_table[or_table['predictor'] != 'const'].rename(columns={
        'predictor': 'label', 'OR': 'estimate',
        'CI_lower': 'ci_lower', 'CI_upper': 'ci_upper',
    })
    fig = ThyroidStatisticalAnalyzer.create_forest_plot(
        forest_df, title='Odds Ratios — Logistic Regression'
    )
    fig.show()

In [ ]:
# Plain-English clinical snippet
if 'error' not in lr_result:
    snippet = ThyroidStatisticalAnalyzer.format_clinical_snippet(
        lr_result, model_type='OR', outcome_label=outcome.replace('_', ' ')
    )
    print(snippet)

## 5. Cox Proportional Hazards

In [ ]:
time_col = THYROID_SURVIVAL['time_col']
event_col = THYROID_SURVIVAL['event_col']

if time_col not in df.columns or event_col not in df.columns:
    print(f'Survival columns not in {resolved}. Try survival_cohort_enriched.')
else:
    cox_predictors = [
        p for p in THYROID_PREDICTORS
        if p in df.columns and p not in (time_col, event_col)
        and pd.api.types.is_numeric_dtype(df[p])
    ][:8]

    cox_result = analyzer.fit_cox_ph(
        time_col=time_col,
        event_col=event_col,
        predictors=cox_predictors,
        data=df,
    )

    if 'error' in cox_result:
        print('ERROR:', cox_result['error'])
    else:
        print(f"N={cox_result['n_obs']:,} | Events={cox_result['n_events']} | "
              f"Concordance={cox_result['concordance']:.4f}")
        for w in cox_result.get('warnings', []):
            print('⚠', w)
        display(cox_result['hr_table'])

In [ ]:
# Forest plot (HR) + clinical snippet
if 'error' not in cox_result:
    hr_table = cox_result['hr_table']
    forest_df = hr_table.rename(columns={
        'covariate': 'label', 'HR': 'estimate',
        'CI_lower': 'ci_lower', 'CI_upper': 'ci_upper',
    })
    fig = ThyroidStatisticalAnalyzer.create_forest_plot(
        forest_df, title='Hazard Ratios — Cox PH Model'
    )
    fig.show()

    snippet = ThyroidStatisticalAnalyzer.format_clinical_snippet(
        cox_result, model_type='HR', outcome_label='disease recurrence'
    )
    print(snippet)

## 6. Longitudinal Tg/TSH — Mixed-Effects Trajectory

In [ ]:
# Requires extracted_clinical_events_v4 or longitudinal_lab_view
long_result = analyzer.longitudinal_summary(marker='tg')

if 'error' in long_result:
    print('Longitudinal Tg:', long_result['error'])
else:
    print(f"N patients: {long_result['n_patients']:,}")
    print(f"N obs: {long_result['n_obs']:,}")
    print(f"Slope β/yr: {long_result['slope']:+.4f}")
    print(f"95% CI: {long_result['slope_ci']}")
    print(f"p-value: {long_result.get('p_value','N/A')}")
    print()
    print('Clinical interpretation:', long_result['clinical_note'])
    print()
    print('Model:', long_result['model_summary'])

In [ ]:
# Per-patient slope distribution
if 'error' not in long_result:
    import plotly.graph_objects as go
    pp = long_result['per_patient_summary']
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=pp['slope_per_day'].dropna() * 365.25,
        nbinsx=40,
        marker_color='#2dd4bf',
        name='Annual slope',
        opacity=0.8,
    ))
    fig.update_layout(
        title='Per-Patient Tg Slope Distribution (log Tg / year)',
        xaxis_title='Slope per year',
        yaxis_title='Patients',
        height=350,
    )
    fig.show()
    print(f"{long_result['rising_pct']:.1f}% of patients have rising Tg trajectory")

In [ ]:
# TSH trajectory
tsh_result = analyzer.longitudinal_summary(marker='tsh')
if 'error' in tsh_result:
    print('TSH:', tsh_result['error'])
else:
    print(f"TSH slope β/yr: {tsh_result['slope']:+.4f} (p={tsh_result.get('p_value','N/A')})")
    print(tsh_result['clinical_note'])

## 7. Power & Sample-Size Analysis

In [ ]:
# H1: BRAF+ vs BRAF− recurrence rate comparison
r1 = ThyroidStatisticalAnalyzer.power_two_proportions(
    p1=0.15, p2=0.05, alpha=0.05, power=0.80
)
print('H1: BRAF+ vs BRAF− recurrence (15% vs 5%)')
print(f"  n per group: {r1['n_per_group']}")
print(f"  n total: {r1['n_total']}")
print(f"  Effect size (Cohen h): {r1['effect_size_h']:.3f} ({r1['effect_label']})")
print(f"  Formula: {r1['formula']}")

In [ ]:
# H2: Logistic regression — ETE → recurrence, OR=2.0, event rate 10%
r2 = ThyroidStatisticalAnalyzer.power_logistic(
    p_event=0.10, or_detect=2.0, alpha=0.05, power=0.80
)
print('H2: ETE → recurrence OR=2.0 logistic regression')
print(f"  n total required: {r2['n_total']}")
print(f"  Baseline event rate: {r2['p_event_baseline']:.0%}")
print(f"  Exposed event rate: {r2['p_event_exposed']:.0%}")

In [ ]:
# H3: Cox log-rank for HR=1.8, 10% event rate
r3 = ThyroidStatisticalAnalyzer.sample_size_km(
    hr=1.8, alpha=0.05, power=0.80, event_rate=0.10
)
print('H3: Cox log-rank — HR=1.8, event rate 10%')
print(f"  Events required: {r3['events_required']}")
print(f"  n total: {r3['n_total']}")
print(f"  n per group: {r3['n_group1']} / {r3['n_group2']}")

In [ ]:
# Sensitivity: power curve across OR range
import plotly.graph_objects as go

or_range = [1.2, 1.5, 1.8, 2.0, 2.5, 3.0]
n_values = [
    ThyroidStatisticalAnalyzer.power_logistic(p_event=0.10, or_detect=or_v)['n_total']
    for or_v in or_range
]

fig = go.Figure(go.Scatter(x=or_range, y=n_values, mode='lines+markers', marker_color='#2dd4bf'))
fig.update_layout(
    title='Required Sample Size vs Target OR (logistic, event rate 10%)',
    xaxis_title='Target Odds Ratio',
    yaxis_title='n total',
    height=350,
)
fig.show()

## 8. Spearman Correlation Matrix

In [ ]:
numeric_predictors = [
    p for p in THYROID_PREDICTORS
    if p in df.columns and pd.api.types.is_numeric_dtype(df[p])
]
print(f'Computing Spearman correlations for {len(numeric_predictors)} predictors')

corr, pval = analyzer.correlation_matrix_with_pvalues(df, numeric_predictors, method='spearman')
fig = ThyroidStatisticalAnalyzer.create_correlation_heatmap(
    corr, pval, title='Spearman Correlation Matrix (* p<0.05, ** p<0.01, *** p<0.001)'
)
fig.show()

## 9. NSQIP Complication Analysis (if complications table available)

In [ ]:
# Check if complications table is available
try:
    n_comp = con.execute("SELECT COUNT(*) FROM complications").fetchone()[0]
    print(f'complications table: {n_comp:,} rows')
    HAS_NSQIP = True
except Exception as e:
    print(f'complications table not available: {e}')
    HAS_NSQIP = False

In [ ]:
if HAS_NSQIP:
    nsqip_sql = """
    WITH base AS (
        SELECT
            mc.research_id,
            mc.age_at_surgery,
            mc.sex,
            TRY_CAST(tp.histology_1_largest_tumor_cm AS DOUBLE) AS largest_tumor_cm,
            CASE
                WHEN LOWER(CAST(comp.rln_injury_or_vocal_cord_paralysis_vocal_cord_palsy AS VARCHAR))
                     NOT IN ('', 'no', 'none', 'n/a', 'na', 'nan', '0', 'false')
                     AND comp.rln_injury_or_vocal_cord_paralysis_vocal_cord_palsy IS NOT NULL
                THEN 1 ELSE 0
            END AS rln_injury,
            CASE
                WHEN LOWER(CAST(comp.hypocalcemia AS VARCHAR))
                     NOT IN ('', 'no', 'none', 'n/a', 'na', 'nan', '0', 'false')
                     AND comp.hypocalcemia IS NOT NULL
                THEN 1 ELSE 0
            END AS hypocalcemia
        FROM master_cohort mc
        INNER JOIN complications comp ON mc.research_id = CAST(comp.research_id AS INT)
        LEFT JOIN tumor_pathology tp ON mc.research_id = tp.research_id
    )
    SELECT *, GREATEST(rln_injury, hypocalcemia) AS any_nsqip_complication
    FROM base
    """
    nsqip_df = con.execute(nsqip_sql).fetchdf()
    print(f'NSQIP cohort: {len(nsqip_df):,} patients with complication data')
    print(f"RLN injury rate: {nsqip_df['rln_injury'].mean():.1%}")
    print(f"Hypocalcemia rate: {nsqip_df['hypocalcemia'].mean():.1%}")
    nsqip_df.head(3)

In [ ]:
if HAS_NSQIP and len(nsqip_df) > 20:
    nsqip_predictors = [p for p in ['age_at_surgery', 'largest_tumor_cm'] if p in nsqip_df.columns]
    comp_result = analyzer.fit_logistic_regression(
        outcome='any_nsqip_complication',
        predictors=nsqip_predictors,
        data=nsqip_df,
    )
    if 'error' not in comp_result:
        display(comp_result['or_table'])
        snippet = ThyroidStatisticalAnalyzer.format_clinical_snippet(
            comp_result, model_type='OR', outcome_label='any NSQIP complication'
        )
        print(snippet)
    else:
        print(comp_result['error'])

## 10. Missing Data Summary

In [ ]:
miss = ThyroidStatisticalAnalyzer.missing_data_summary(df)
print(f"{(miss['pct_missing'] > 0).sum()} / {len(miss)} columns have missing data")
miss[miss['pct_missing'] > 0].head(20)